In [1]:
print(0)

0


In [2]:
import os
import json
import time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

try:
    import timm
except Exception as e:
    raise ImportError("timm is required. Install with: pip install timm") from e

In [3]:
TEST_CONFIG = {
    "test_folder_path": "/path/to/test_dataset",  # CHANGE THIS
    "results_saving_folder": "/path/to/test_results",  # CHANGE THIS
    "trained_models_root": "/home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models",
    "training_results_root": "/home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_results",
    "img_size": 224,
    "batch_size": 32,
    "num_workers": 4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "threshold": 0.5,  # classification threshold
    "model_name": "vit_tiny_patch16_224",
    "datasets": ["bone", "skin"],  # must match training
}

In [4]:
class TestImageFolder(Dataset):
    """Dataset for test images without labels."""
    def __init__(self, file_paths, img_size=224):
        self.file_paths = list(file_paths)
        self.img_size = img_size
        
        self.transform = transforms.Compose([
            transforms.Resize(int(img_size * 1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]),
        ])
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        p = self.file_paths[idx]
        try:
            img = Image.open(p).convert("RGB")
        except Exception as e:
            print(f"Error loading {p}: {e}")
            img = Image.new("RGB", (self.img_size, self.img_size), (0, 0, 0))
        
        img = self.transform(img)
        return img, str(p)

In [5]:
def create_model(num_classes=2, model_name="vit_tiny_patch16_224", pretrained=False):
    """Create model architecture (same as training)."""
    try:
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    except Exception:
        model = timm.create_model("vit_small_patch16_224", pretrained=pretrained, num_classes=num_classes)
    return model

In [6]:
def load_best_model(dataset_name, class_name, models_root, device, model_name):
    models_path = Path(models_root)
    pattern = f"{dataset_name}_{class_name}_fold*.pth"
    model_files = list(models_path.glob(pattern))
    
    if len(model_files) == 0:
        raise FileNotFoundError(f"No models found for {dataset_name}/{class_name}")
    
    # Load each model and find the one with best validation performance
    best_model = None
    best_score = -float('inf')
    best_model_info = None
    
    for model_file in model_files:
        try:
            checkpoint = torch.load(model_file, map_location=device)
            
            # Create model and load weights
            model = create_model(num_classes=2, model_name=model_name, pretrained=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            model.to(device)
            model.eval()
            
            # Try to get validation score from corresponding results
            fold_num = model_file.stem.split('fold')[-1].split('_')[0]
            
            # Default to using this model if we can't find scores
            if best_model is None:
                best_model = model
                best_model_info = {
                    'path': str(model_file),
                    'fold': fold_num,
                    'epoch': checkpoint.get('epoch', 'unknown')
                }
            
        except Exception as e:
            print(f"Error loading model {model_file}: {e}")
            continue
    
    if best_model is None:
        raise ValueError(f"Could not load any model for {dataset_name}/{class_name}")
    
    return best_model, best_model_info

In [7]:
@torch.no_grad()
def predict_batch(model, loader, device):
    """Run inference on a batch of images."""
    model.eval()
    all_probs = []
    all_paths = []
    
    for imgs, paths in tqdm(loader, desc="Predicting", leave=False):
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # probability of class 1 (plausible)
        
        all_probs.extend(probs.tolist())
        all_paths.extend(paths)
    
    return np.array(all_probs), all_paths

In [8]:
def test_class(dataset_name, class_name, test_folder, cfg):
    """Test a single class and return detailed results."""
    print(f"\n{'='*70}")
    print(f"Testing: {dataset_name}/{class_name}")
    print(f"{'='*70}")
    
    # Find test images
    class_path = Path(test_folder) / dataset_name / class_name
    if not class_path.exists():
        print(f"Path not found: {class_path}")
        return None
    
    # Collect all image files
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    image_files = []
    for ext in image_extensions:
        image_files.extend(class_path.rglob(f"*{ext}"))
        image_files.extend(class_path.rglob(f"*{ext.upper()}"))
    
    if len(image_files) == 0:
        print(f"No images found in {class_path}")
        return None
    
    print(f"Found {len(image_files)} images")
    
    # Load model
    try:
        model, model_info = load_best_model(
            dataset_name, class_name, 
            cfg['trained_models_root'], 
            cfg['device'],
            cfg['model_name']
        )
        print(f"Loaded model from: {model_info['path']}")
        print(f"Model fold: {model_info['fold']}, Epoch: {model_info['epoch']}")
    except Exception as e:
        print(f"Error loading model: {e}")
        return None
    
    # Create dataset and loader
    test_dataset = TestImageFolder([str(f) for f in image_files], img_size=cfg['img_size'])
    test_loader = DataLoader(
        test_dataset, 
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=cfg['num_workers'],
        pin_memory=True
    )
    
    # Run inference
    probs, paths = predict_batch(model, test_loader, cfg['device'])
    predictions = (probs >= cfg['threshold']).astype(int)
    
    # Count results
    n_plausible = int(np.sum(predictions == 1))
    n_implausible = int(np.sum(predictions == 0))
    n_total = len(predictions)
    plausible_ratio = n_plausible / n_total if n_total > 0 else 0.0
    
    # Create detailed results
    results = {
        'dataset': dataset_name,
        'class': class_name,
        'model_info': model_info,
        'test_stats': {
            'total_images': n_total,
            'plausible_count': n_plausible,
            'implausible_count': n_implausible,
            'plausible_ratio': plausible_ratio,
            'mean_probability': float(np.mean(probs)),
            'std_probability': float(np.std(probs)),
            'median_probability': float(np.median(probs)),
            'min_probability': float(np.min(probs)),
            'max_probability': float(np.max(probs))
        },
        'predictions': {
            'paths': paths,
            'probabilities': probs.tolist(),
            'predictions': predictions.tolist()
        }
    }
    
    # Load training metrics if available
    try:
        results_path = Path(cfg['training_results_root'])
        summary_file = results_path / f"{dataset_name}_{class_name}_summary.json"
        if summary_file.exists():
            with open(summary_file, 'r') as f:
                training_summary = json.load(f)
                results['training_metrics'] = training_summary.get('aggregate', [])
    except Exception as e:
        print(f"Could not load training metrics: {e}")
    
    print(f"\nResults:")
    print(f"  Total images: {n_total}")
    print(f"  Plausible: {n_plausible} ({plausible_ratio:.2%})")
    print(f"  Implausible: {n_implausible} ({1-plausible_ratio:.2%})")
    print(f"  Mean probability: {results['test_stats']['mean_probability']:.4f}")
    
    return results

In [9]:
def run_all_tests(cfg):
    """Run tests on all dataset/class combinations."""
    print("\n" + "="*70)
    print("STARTING COMPREHENSIVE MODEL TESTING")
    print("="*70)
    print(f"Test folder: {cfg['test_folder_path']}")
    print(f"Results folder: {cfg['results_saving_folder']}")
    print(f"Device: {cfg['device']}")
    
    # Create results directory
    results_dir = Path(cfg['results_saving_folder'])
    results_dir.mkdir(parents=True, exist_ok=True)
    
    test_folder = Path(cfg['test_folder_path'])
    if not test_folder.exists():
        raise FileNotFoundError(f"Test folder not found: {test_folder}")
    
    # Collect all results
    all_results = []
    summary_records = []
    
    for dataset_name in cfg['datasets']:
        dataset_path = test_folder / dataset_name
        if not dataset_path.exists():
            print(f"\nDataset folder not found: {dataset_path}")
            continue
        
        # Find all class directories
        class_dirs = [d for d in dataset_path.iterdir() if d.is_dir()]
        
        for class_dir in class_dirs:
            class_name = class_dir.name
            
            try:
                result = test_class(dataset_name, class_name, test_folder, cfg)
                if result is not None:
                    all_results.append(result)
                    
                    # Create summary record
                    summary_record = {
                        'dataset': dataset_name,
                        'class': class_name,
                        'model_path': result['model_info']['path'],
                        'model_fold': result['model_info']['fold'],
                        'model_epoch': result['model_info']['epoch'],
                        'total_images': result['test_stats']['total_images'],
                        'plausible_count': result['test_stats']['plausible_count'],
                        'implausible_count': result['test_stats']['implausible_count'],
                        'plausible_ratio': result['test_stats']['plausible_ratio'],
                        'mean_probability': result['test_stats']['mean_probability'],
                        'std_probability': result['test_stats']['std_probability'],
                        'median_probability': result['test_stats']['median_probability'],
                    }
                    
                    # Add training metrics if available
                    if 'training_metrics' in result:
                        for metric in result['training_metrics']:
                            metric_name = metric.get('metric', '')
                            if metric_name:
                                summary_record[f'train_{metric_name}_mean'] = metric.get('mean', None)
                                summary_record[f'train_{metric_name}_std'] = metric.get('std', None)
                    
                    summary_records.append(summary_record)
                    
            except Exception as e:
                print(f"\nError testing {dataset_name}/{class_name}: {e}")
                import traceback
                traceback.print_exc()
                continue
    
    # =====================================================================
    # SAVE RESULTS
    # =====================================================================
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    
    # 1. Save complete results JSON
    complete_results_file = results_dir / f"complete_test_results_{timestamp}.json"
    with open(complete_results_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"\n✓ Saved complete results: {complete_results_file}")
    
    # 2. Save summary CSV
    if len(summary_records) > 0:
        summary_df = pd.DataFrame(summary_records)
        summary_csv = results_dir / f"test_summary_{timestamp}.csv"
        summary_df.to_csv(summary_csv, index=False)
        print(f"✓ Saved summary CSV: {summary_csv}")
        
        # 3. Create detailed summary table
        print("\n" + "="*70)
        print("TEST RESULTS SUMMARY")
        print("="*70)
        print(summary_df.to_string(index=False))
        
        # 4. Save per-image predictions for each class
        for result in all_results:
            dataset = result['dataset']
            class_name = result['class']
            
            pred_df = pd.DataFrame({
                'image_path': result['predictions']['paths'],
                'probability_plausible': result['predictions']['probabilities'],
                'prediction': ['plausible' if p == 1 else 'implausible' 
                              for p in result['predictions']['predictions']]
            })
            
            pred_file = results_dir / f"predictions_{dataset}_{class_name}_{timestamp}.csv"
            pred_df.to_csv(pred_file, index=False)
            print(f"✓ Saved predictions: {pred_file}")
    
    # 5. Generate aggregate statistics
    if len(summary_records) > 0:
        aggregate_stats = {
            'total_classes_tested': len(summary_records),
            'total_images_tested': sum(r['total_images'] for r in summary_records),
            'total_plausible': sum(r['plausible_count'] for r in summary_records),
            'total_implausible': sum(r['implausible_count'] for r in summary_records),
            'average_plausible_ratio': np.mean([r['plausible_ratio'] for r in summary_records]),
            'std_plausible_ratio': np.std([r['plausible_ratio'] for r in summary_records]),
        }
        
        # Per-dataset statistics
        dataset_stats = {}
        for dataset in cfg['datasets']:
            dataset_records = [r for r in summary_records if r['dataset'] == dataset]
            if len(dataset_records) > 0:
                dataset_stats[dataset] = {
                    'n_classes': len(dataset_records),
                    'total_images': sum(r['total_images'] for r in dataset_records),
                    'avg_plausible_ratio': np.mean([r['plausible_ratio'] for r in dataset_records]),
                }
        
        aggregate_stats['per_dataset'] = dataset_stats
        
        agg_file = results_dir / f"aggregate_statistics_{timestamp}.json"
        with open(agg_file, 'w') as f:
            json.dump(aggregate_stats, f, indent=2)
        print(f"✓ Saved aggregate statistics: {agg_file}")
        
        print("\n" + "="*70)
        print("AGGREGATE STATISTICS")
        print("="*70)
        print(json.dumps(aggregate_stats, indent=2))
    
    print("\n" + "="*70)
    print("TESTING COMPLETED SUCCESSFULLY")
    print("="*70)
    
    return all_results, summary_df if len(summary_records) > 0 else None

In [10]:
if __name__ == "__main__":
    # UPDATE THESE PATHS BEFORE RUNNING
    TEST_CONFIG["test_folder_path"] = "/home/pratyushprasad/DPO_PROJECT/12.classification_task/test_dataset_vanilla"
    TEST_CONFIG["results_saving_folder"] = "/home/pratyushprasad/DPO_PROJECT/12.classification_task/test_results_1/vanilla"
    
    try:
        all_results, summary_df = run_all_tests(TEST_CONFIG)
        print("\n✓ All tests completed successfully!")
    except Exception as e:
        print(f"\n✗ Error during testing: {e}")
        import traceback
        traceback.print_exc()


STARTING COMPREHENSIVE MODEL TESTING
Test folder: /home/pratyushprasad/DPO_PROJECT/12.classification_task/test_dataset_vanilla
Results folder: /home/pratyushprasad/DPO_PROJECT/12.classification_task/test_results_1/vanilla
Device: cuda

Testing: bone/PLM
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_PLM_fold1.pth
Model fold: 1, Epoch: 1



Results:
  Total images: 100
  Plausible: 87 (87.00%)
  Implausible: 13 (13.00%)
  Mean probability: 0.6785

Testing: bone/PEB
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_PEB_fold1_1761903428.pth
Model fold: 1, Epoch: 2



Results:
  Total images: 100
  Plausible: 47 (47.00%)
  Implausible: 53 (53.00%)
  Mean probability: 0.4696

Testing: bone/MYB
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_MYB_fold4_1761903759.pth
Model fold: 4, Epoch: 2



Results:
  Total images: 100
  Plausible: 50 (50.00%)
  Implausible: 50 (50.00%)
  Mean probability: 0.5148

Testing: bone/NGS
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_NGS_fold5.pth
Model fold: 5, Epoch: 1



Results:
  Total images: 100
  Plausible: 31 (31.00%)
  Implausible: 69 (69.00%)
  Mean probability: 0.4327

Testing: bone/BLA
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_BLA_fold1_1761904054.pth
Model fold: 1, Epoch: 5



Results:
  Total images: 100
  Plausible: 36 (36.00%)
  Implausible: 64 (64.00%)
  Mean probability: 0.3996

Testing: bone/OTH
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_OTH_fold3.pth
Model fold: 3, Epoch: 1


... (15 outputs left out)


Results:
  Total images: 100
  Plausible: 18 (18.00%)
  Implausible: 82 (82.00%)
  Mean probability: 0.2659

Testing: skin/bkl
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/skin_bkl_fold1_1761907691.pth
Model fold: 1, Epoch: 4



Results:
  Total images: 100
  Plausible: 2 (2.00%)
  Implausible: 98 (98.00%)
  Mean probability: 0.1545

Testing: skin/nv
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/skin_nv_fold5.pth
Model fold: 5, Epoch: 1



Results:
  Total images: 100
  Plausible: 26 (26.00%)
  Implausible: 74 (74.00%)
  Mean probability: 0.4236

Testing: skin/mel
Found 100 images
Loaded model from: /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/skin_mel_fold5_1761908315.pth
Model fold: 5, Epoch: 2



Results:
  Total images: 100
  Plausible: 17 (17.00%)
  Implausible: 83 (83.00%)
  Mean probability: 0.3663

✓ Saved complete results: /home/pratyushprasad/DPO_PROJECT/12.classification_task/test_results_1/vanilla/complete_test_results_20251031_114327.json
✓ Saved summary CSV: /home/pratyushprasad/DPO_PROJECT/12.classification_task/test_results_1/vanilla/test_summary_20251031_114327.csv

TEST RESULTS SUMMARY
dataset class                                                                                         model_path model_fold  model_epoch  total_images  plausible_count  implausible_count  plausible_ratio  mean_probability  std_probability  median_probability  train_accuracy_mean  train_accuracy_std  train_precision_mean  train_precision_std  train_recall_mean  train_recall_std  train_f1_mean  train_f1_std  train_auc_mean  train_auc_std
   bone   PLM            /home/pratyushprasad/DPO_PROJECT/12.classification_task/saved_models/bone_PLM_fold1.pth          1            1           